<a href="https://colab.research.google.com/github/raisharad/GenerativeAIandAgenticAI/blob/main/SharadRai_Demo1_NLP_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demo 1 - Pre-processing + Linguistic Analysis
## NLP Session - Part 1

---

**What we will do in this demo:**

We take a real IMDb movie review - plain text - and run it through a full NLP pipeline, one step at a time.

By the end you will have seen exactly what happens inside every NLP system when it reads text:

1. Sentence segmentation
2. Tokenization
3. Regex cleaning
4. Part-of-Speech (POS) tagging
5. Dependency parsing
6. Chunking - noun phrase extraction
7. Lemmatization
8. Named Entity Recognition (bonus step)

**Library we use:** `spaCy` - a professional NLP library that handles all of the above in one line.


---
## Step 0 - Install and Load spaCy

We need to install spaCy and download its English language model.

The language model is a pre-trained file that already knows English grammar, common words, and how to tag parts of speech. Think of it as the dictionary and grammar textbook that spaCy uses internally.

In [1]:
# Install spaCy - run this cell once
# The exclamation mark means: run as a terminal command, not Python
!pip install spacy --quiet

# Download the small English language model
# en_core_web_sm = English, trained on web text, small size
!python -m spacy download en_core_web_sm --quiet

print("Done. spaCy and the English model are ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 74.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Done. spaCy and the English model are ready.


In [2]:
import spacy

# Load the English model into a variable called nlp
# When we call nlp(some_text) it runs the full pipeline automatically
nlp = spacy.load("en_core_web_sm")

print("spaCy version:", spacy.__version__)
print("Model loaded successfully.")
print()
print("Pipeline steps active:", nlp.pipe_names)
print("These are the steps spaCy runs internally on every text you give it.")

spaCy version: 3.8.14
Model loaded successfully.

Pipeline steps active: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
These are the steps spaCy runs internally on every text you give it.


---
## Our Input - Review

We will use this review for every step in this notebook.

Notice it has:
- Multiple sentences
- A mix of uppercase and lowercase
- Punctuation including exclamation marks
- Different word forms: was, are, loved, delivers
- An HTML tag `<br>` - this is common in real web-scraped data

In [3]:
raw_review = (
    "The Shawshank Redemption is one of the greatest films ever made.<br> "
    "Tim Robbins and Morgan Freeman deliver performances that are absolutely breathtaking. "
    "The story of hope and friendship in an American prison kept me on the edge of my seat. "
    "I loved every single scene. This movie deserves all the awards it has ever received!!!"
)

print("Raw review text:")
print("-" * 60)
print(raw_review)
print("-" * 60)
print("Total characters:", len(raw_review))

Raw review text:
------------------------------------------------------------
The Shawshank Redemption is one of the greatest films ever made.<br> Tim Robbins and Morgan Freeman deliver performances that are absolutely breathtaking. The story of hope and friendship in an American prison kept me on the edge of my seat. I loved every single scene. This movie deserves all the awards it has ever received!!!
------------------------------------------------------------
Total characters: 328


---
## Step 1 - Sentence Segmentation

**What is it?**

Sentence segmentation means splitting a block of text into individual sentences.

This sounds simple but it is trickier than it looks. Consider:

- `"Dr. Smith works at St. Mary's hospital."` - the full stop after Dr is NOT a sentence boundary
- `"I loved it. The ending was perfect."` - the full stop after it IS a sentence boundary

spaCy uses a trained model to figure this out correctly.

**Why do we need it?**

Many NLP tasks - summarization, translation, sentiment analysis - work at the sentence level. We need to split the paragraph into sentences first.

In [4]:
import re

# Remove the HTML <br> tag before passing text to spaCy
# re.sub(pattern, replacement, text) finds all matches and replaces them
cleaned_text = re.sub(r"<br>", " ", raw_review)
cleaned_text = cleaned_text.strip()

print("After removing the HTML tag:")
print(cleaned_text)

After removing the HTML tag:
The Shawshank Redemption is one of the greatest films ever made.  Tim Robbins and Morgan Freeman deliver performances that are absolutely breathtaking. The story of hope and friendship in an American prison kept me on the edge of my seat. I loved every single scene. This movie deserves all the awards it has ever received!!!


In [5]:
# Pass the cleaned text to spaCy
# This single line runs the ENTIRE pipeline: tokenizer, tagger, parser, NER
doc = nlp(cleaned_text)

# doc.sents is a generator that yields sentence objects
sentences = list(doc.sents)

print(f"spaCy found {len(sentences)} sentences:")
print()
for i, sentence in enumerate(sentences, start=1):
    print(f"  Sentence {i}: {sentence.text.strip()}")

spaCy found 5 sentences:

  Sentence 1: The Shawshank Redemption is one of the greatest films ever made.
  Sentence 2: Tim Robbins and Morgan Freeman deliver performances that are absolutely breathtaking.
  Sentence 3: The story of hope and friendship in an American prison kept me on the edge of my seat.
  Sentence 4: I loved every single scene.
  Sentence 5: This movie deserves all the awards it has ever received!!!


In [6]:
# We will use Sentence 2 for the remaining steps
# It has a good variety of word types

working_sentence = sentences[1]

print("Working sentence for the rest of Demo 1:")
print()
print(" ", working_sentence.text.strip())

Working sentence for the rest of Demo 1:

  Tim Robbins and Morgan Freeman deliver performances that are absolutely breathtaking.


---
## Step 2 - Tokenization

**What is it?**

Tokenization splits text into individual units called **tokens**. In most cases a token is a word, but punctuation marks are also separate tokens.

**Example:**

```
"I loved it!"  -->  ["I",  "loved",  "it",  "!"]
```

Machines cannot work with raw character strings. They need individual units. Tokenization creates those units.

**Why is punctuation a separate token?**

Because `"it!"` and `"it"` should be treated as the same word. Separating the `!` lets us handle the word and the punctuation independently.

In [7]:
# spaCy tokenizes automatically when you call nlp()
# We just iterate over the sentence to see the tokens

tokens = [token.text for token in working_sentence]

print("Tokens in the working sentence:")
print()
for i, t in enumerate(tokens):
    print(f"  Token {i+1:>2}: {repr(t)}")

print()
print(f"Total number of tokens: {len(tokens)}")

Tokens in the working sentence:

  Token  1: 'Tim'
  Token  2: 'Robbins'
  Token  3: 'and'
  Token  4: 'Morgan'
  Token  5: 'Freeman'
  Token  6: 'deliver'
  Token  7: 'performances'
  Token  8: 'that'
  Token  9: 'are'
  Token 10: 'absolutely'
  Token 11: 'breathtaking'
  Token 12: '.'

Total number of tokens: 12


In [8]:
print("Token breakdown:")
print()
print(f"{"Token":<22} {"Is a word?":<15} {"Is punctuation?":<18} {"Is a stopword?"}")
print("-" * 70)

for token in working_sentence:
    is_word = not token.is_punct and not token.is_space
    print(f"{token.text:<22} {str(is_word):<15} {str(token.is_punct):<18} {token.is_stop}")

print()
print("Stopwords are common words like the, and, are that carry little meaning.")
print("We often remove them before training a model.")

Token breakdown:

Token                  Is a word?      Is punctuation?    Is a stopword?
----------------------------------------------------------------------
Tim                    True            False              False
Robbins                True            False              False
and                    True            False              True
Morgan                 True            False              False
Freeman                True            False              False
deliver                True            False              False
performances           True            False              False
that                   True            False              True
are                    True            False              True
absolutely             True            False              False
breathtaking           True            False              False
.                      False           True               False

Stopwords are common words like the, and, are that carry little meaning.

---
## Step 3 - Regex Cleaning

**What is regex?**

Regex stands for **Regular Expressions** - a way to describe patterns in text.

| Pattern | What it matches |
|---------|----------------|
| `\\d+` | One or more digits such as 2024 or 42 |
| `[^a-z ]` | Anything that is NOT a lowercase letter or space |
| `http\\S+` | A URL starting with http |
| ` +` | Two or more spaces in a row |

**Why clean text?**

Real web text contains noise - HTML tags, URLs, numbers, extra punctuation. Without cleaning, `"film"` and `"film!!!"` look like two different words to a model.

**What we clean:**
1. Lowercase everything so `The` and `the` become the same word
2. Remove numbers
3. Remove punctuation and special characters
4. Collapse multiple spaces into one

In [9]:
sample = working_sentence.text.strip()
print("Starting text:")
print(" ", sample)
print()

# Step A: lowercase
step_a = sample.lower()
print("After lowercasing:")
print(" ", step_a)
print()

# Step B: remove digits  (\d+ matches one or more digits)
step_b = re.sub(r"\d+", "", step_a)
print("After removing digits:")
print(" ", step_b)
print()

# Step C: remove punctuation  ([^a-z ] keeps only letters and spaces)
step_c = re.sub(r"[^a-z ]", "", step_b)
print("After removing punctuation:")
print(" ", step_c)
print()

# Step D: collapse extra spaces
step_d = re.sub(r" +", " ", step_c).strip()
print("After collapsing spaces:")
print(" ", step_d)

Starting text:
  Tim Robbins and Morgan Freeman deliver performances that are absolutely breathtaking.

After lowercasing:
  tim robbins and morgan freeman deliver performances that are absolutely breathtaking.

After removing digits:
  tim robbins and morgan freeman deliver performances that are absolutely breathtaking.

After removing punctuation:
  tim robbins and morgan freeman deliver performances that are absolutely breathtaking

After collapsing spaces:
  tim robbins and morgan freeman deliver performances that are absolutely breathtaking


In [10]:
def clean_text(text):
    """Cleans raw text: lowercase, remove digits, remove punctuation, collapse spaces."""
    text = text.lower()
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^a-z ]", "", text)
    text = re.sub(r" +", " ", text).strip()
    return text


print("Cleaned version of all sentences:")
print()
for i, sent in enumerate(sentences, start=1):
    print(f"  Sentence {i}: {clean_text(sent.text)}")

Cleaned version of all sentences:

  Sentence 1: the shawshank redemption is one of the greatest films ever made
  Sentence 2: tim robbins and morgan freeman deliver performances that are absolutely breathtaking
  Sentence 3: the story of hope and friendship in an american prison kept me on the edge of my seat
  Sentence 4: i loved every single scene
  Sentence 5: this movie deserves all the awards it has ever received


---
## Step 4 - Part-of-Speech (POS) Tagging

**What is it?**

POS tagging assigns a grammatical label to each word.

| Tag | Meaning | Example |
|-----|---------|--------|
| NOUN | Noun | film, seat, story |
| VERB | Verb | deliver, loved, made |
| ADJ | Adjective | breathtaking, greatest |
| ADV | Adverb | absolutely, ever |
| PROPN | Proper Noun | Tim Robbins, Shawshank |
| DET | Determiner | the, a, an |
| ADP | Preposition | in, of, on |
| PUNCT | Punctuation | . , ! |

**Why does this matter?**

The word `"film"` is a noun in `"I watched a film"` but a verb in `"film the scene"`. Same spelling, different role. POS tagging resolves this ambiguity using surrounding context.

In [11]:
# token.pos_ gives the simple category: NOUN, VERB, ADJ...
# spacy.explain() returns a plain-English description of any tag

print("POS tags for the working sentence:")
print()
print(f"{"Token":<22} {"POS Tag":<12} {"What it means"}")
print("-" * 60)

for token in working_sentence:
    explanation = spacy.explain(token.pos_) or ""
    print(f"{token.text:<22} {token.pos_:<12} {explanation}")

POS tags for the working sentence:

Token                  POS Tag      What it means
------------------------------------------------------------
Tim                    PROPN        proper noun
Robbins                PROPN        proper noun
and                    CCONJ        coordinating conjunction
Morgan                 PROPN        proper noun
Freeman                PROPN        proper noun
deliver                VERB         verb
performances           NOUN         noun
that                   PRON         pronoun
are                    AUX          auxiliary
absolutely             ADV          adverb
breathtaking           ADJ          adjective
.                      PUNCT        punctuation


In [12]:
# Extract all adjectives from the full review
# Adjectives carry strong sentiment signal

adjectives = [token.text for token in doc if token.pos_ == "ADJ"]

print("All adjectives in the review:")
print()
print(adjectives)
print()
print("These words alone give a strong signal that the review is positive.")
print("A model that sees breathtaking and greatest will likely predict POSITIVE.")

All adjectives in the review:

['greatest', 'breathtaking', 'American', 'single']

These words alone give a strong signal that the review is positive.
A model that sees breathtaking and greatest will likely predict POSITIVE.


In [13]:
from collections import Counter

pos_counts = Counter(
    token.pos_
    for token in doc
    if not token.is_punct and not token.is_space
)

print("POS tag counts across the full review:")
print()
for tag, count in pos_counts.most_common():
    explanation = spacy.explain(tag) or tag
    print(f"  {tag:<8} {count:>3} occurrences   ({explanation})")

POS tag counts across the full review:

  NOUN      11 occurrences   (noun)
  DET        9 occurrences   (determiner)
  PROPN      6 occurrences   (proper noun)
  VERB       6 occurrences   (verb)
  ADP        5 occurrences   (adposition)
  PRON       5 occurrences   (pronoun)
  ADJ        4 occurrences   (adjective)
  AUX        3 occurrences   (auxiliary)
  ADV        3 occurrences   (adverb)
  CCONJ      2 occurrences   (coordinating conjunction)
  NUM        1 occurrences   (numeral)


---
## Step 5 - Dependency Parsing

**What is it?**

Dependency parsing draws connections between words to show which word depends on which other word.

Every sentence has a **root** - usually the main verb. Everything else connects to the root or to another word.

**Example:**

```
"Tim Robbins delivers great performances."

delivers             <-- ROOT (the main verb)
  |-- Tim Robbins    (nsubj: who is doing the delivering?)
  |-- performances   (dobj: what is being delivered?)
         |-- great   (amod: what kind of performances?)
```

**Why does this matter?**

Consider: `"The dog bit the man"` vs `"The man bit the dog"`. Same words, different structure, opposite meaning. Dependency parsing captures WHO is doing WHAT to WHOM.

In [14]:
# token.dep_  = the type of dependency relation (nsubj, dobj, amod...)
# token.head  = the word this token connects to

print("Dependency parse for the working sentence:")
print()
print(f"{"Token":<22} {"Relation":<18} {"Head word":<18} {"Relation meaning"}")
print("-" * 80)

for token in working_sentence:
    meaning = spacy.explain(token.dep_) or ""
    print(f"{token.text:<22} {token.dep_:<18} {token.head.text:<18} {meaning}")

Dependency parse for the working sentence:

Token                  Relation           Head word          Relation meaning
--------------------------------------------------------------------------------
Tim                    compound           Robbins            compound
Robbins                nsubj              deliver            nominal subject
and                    cc                 Robbins            coordinating conjunction
Morgan                 compound           Freeman            compound
Freeman                conj               Robbins            conjunct
deliver                ROOT               deliver            root
performances           dobj               deliver            direct object
that                   nsubj              are                nominal subject
are                    relcl              performances       relative clause modifier
absolutely             advmod             breathtaking       adverbial modifier
breathtaking           acomp            

In [15]:
print("Key structural roles in the sentence:")
print()

for token in working_sentence:
    if token.dep_ == "ROOT":
        print(f"  Main verb (ROOT) : {token.text}")
    if token.dep_ == "nsubj":
        print(f"  Subject          : {token.text}  (the one performing the action)")
    if token.dep_ == "dobj":
        print(f"  Direct object    : {token.text}  (the one receiving the action)")
    if token.dep_ == "amod":
        print(f"  Adjective        : {token.text}  modifies  {token.head.text}")

Key structural roles in the sentence:

  Subject          : Robbins  (the one performing the action)
  Main verb (ROOT) : deliver
  Direct object    : performances  (the one receiving the action)
  Subject          : that  (the one performing the action)


In [16]:
# displacy renders a dependency arc diagram inside the notebook

from spacy import displacy

# Use a short sentence so the diagram is easy to read
short_doc = nlp("Tim Robbins delivers breathtaking performances.")

displacy.render(short_doc, style="dep", jupyter=True)

In [17]:
from spacy import displacy

# Use a short sentence so the diagram is easy to read
short_doc = nlp("The dog chased the cat")

displacy.render(short_doc, style="dep", jupyter=True)

**How to read the diagram:**

- Each word appears at the bottom as a box
- Arrows point from a word to the word it depends on
- The label on each arrow is the type of relationship
- `nsubj` = nominal subject - who is doing the action
- `dobj` = direct object - what receives the action
- `amod` = adjectival modifier - an adjective describing a noun
- The word with no incoming arrow is the ROOT (main verb)

---
## Step 6 - Chunking (Noun Phrase Extraction)

**What is it?**

Chunking groups consecutive words that belong together into meaningful phrases.

The most common type is **noun phrase chunking** - finding groups like:

- `"the greatest films"` rather than just `films`
- `"breathtaking performances"` rather than just `performances`
- `"the edge of my seat"` rather than just `seat`

**Why is this better than single words?**

The phrase `"greatest films"` carries far more meaning than the word `"films"` alone. Chunking preserves this context.

In spaCy, noun phrases are accessed via `doc.noun_chunks`.

In [18]:
print("Noun phrases found in the full review:")
print()
print(f"{"Noun Phrase":<40} {"Root Word":<20} {"Root POS"}")
print("-" * 70)

for chunk in doc.noun_chunks:
    print(f"{chunk.text:<40} {chunk.root.text:<20} {chunk.root.pos_}")

Noun phrases found in the full review:

Noun Phrase                              Root Word            Root POS
----------------------------------------------------------------------
The Shawshank Redemption                 Redemption           PROPN
the greatest films                       films                NOUN
Tim Robbins                              Robbins              PROPN
Morgan Freeman                           Freeman              PROPN
performances                             performances         NOUN
that                                     that                 PRON
The story                                story                NOUN
hope                                     hope                 NOUN
friendship                               friendship           NOUN
an American prison                       prison               NOUN
me                                       me                   PRON
the edge                                 edge                 NOUN
my seat    

In [19]:
# Compare single words vs noun phrases on one sentence

sentence_text = "Tim Robbins and Morgan Freeman deliver breathtaking performances."
sentence_doc = nlp(sentence_text)

# Single content words - lose context
single_words = [
    token.text for token in sentence_doc
    if not token.is_punct and not token.is_space and not token.is_stop
]

# Noun phrases - keep context
noun_phrases = [chunk.text for chunk in sentence_doc.noun_chunks]

print("Sentence:")
print(" ", sentence_text)
print()
print("Single content words - loses context:")
print(" ", single_words)
print()
print("Noun phrases - keeps meaningful groupings:")
print(" ", noun_phrases)
print()
print("With noun phrases we know Tim Robbins is one person,")
print("and breathtaking performances is one idea - not separate words.")

Sentence:
  Tim Robbins and Morgan Freeman deliver breathtaking performances.

Single content words - loses context:
  ['Tim', 'Robbins', 'Morgan', 'Freeman', 'deliver', 'breathtaking', 'performances']

Noun phrases - keeps meaningful groupings:
  ['Tim Robbins', 'Morgan Freeman', 'breathtaking performances']

With noun phrases we know Tim Robbins is one person,
and breathtaking performances is one idea - not separate words.


---
## Step 7 - Lemmatization

**What is it?**

Lemmatization reduces each word to its base dictionary form, called its **lemma**.

| Original word | Lemma |
|--------------|-------|
| running | run |
| loved | love |
| performances | performance |
| greatest | great |
| received | receive |
| delivers | deliver |

**Why is this important?**

Without lemmatization a model treats `"love"`, `"loved"`, and `"loving"` as three different words. With lemmatization they all become `"love"` - one concept. This reduces vocabulary size and helps the model generalise.

**Lemmatization vs Stemming:**

| | Stemming | Lemmatization |
|--|---------|---------------|
| How it works | Chops off endings with rules | Uses a dictionary |
| Output | May not be a real word: `studi` | Always a real word: `study` |
| Accuracy | Lower | Higher |

We use lemmatization for quality NLP work.

In [20]:
# token.lemma_ gives the base form of each word

print("Lemmatization of the working sentence:")
print()

print(f"{'Original word':<22} {'Lemma':<22} {'Changed?'}")
print("-" * 60)

for token in working_sentence:

    # skip punctuation and spaces
    if token.is_punct or token.is_space:
        continue

    # compare original word with lemma
    changed = "YES <-- different" if token.text != token.lemma_ else "same"

    print(f"{token.text:<22} {token.lemma_:<22} {changed}")

Lemmatization of the working sentence:

Original word          Lemma                  Changed?
------------------------------------------------------------
Tim                    Tim                    same
Robbins                Robbins                same
and                    and                    same
Morgan                 Morgan                 same
Freeman                Freeman                same
deliver                deliver                same
performances           performance            YES <-- different
that                   that                   same
are                    be                     YES <-- different
absolutely             absolutely             same
breathtaking           breathtaking           same


In [21]:
def lemmatize_text(spacy_doc):
    """Returns clean lemmas - punctuation, spaces, and stopwords removed."""
    lemmas = []
    for token in spacy_doc:
        if token.is_punct:  continue
        if token.is_space:  continue
        if token.is_stop:   continue
        lemmas.append(token.lemma_.lower())
    return lemmas


lemmatized = lemmatize_text(doc)

total_raw = len([t for t in doc if not t.is_punct and not t.is_space])

print("Lemmatized tokens from the full review (stopwords removed):")
print()
print(lemmatized)
print()
print(f"Raw word count:              {total_raw}")
print(f"After lemmatization:         {len(lemmatized)} meaningful tokens")
print()
print("These tokens are what we pass to a machine learning model in Demo 2.")

Lemmatized tokens from the full review (stopwords removed):

['shawshank', 'redemption', 'great', 'film', 'tim', 'robbins', 'morgan', 'freeman', 'deliver', 'performance', 'absolutely', 'breathtaking', 'story', 'hope', 'friendship', 'american', 'prison', 'keep', 'edge', 'seat', 'love', 'single', 'scene', 'movie', 'deserve', 'award', 'receive']

Raw word count:              55
After lemmatization:         27 meaningful tokens

These tokens are what we pass to a machine learning model in Demo 2.


---
## Step 8 - Named Entity Recognition (Bonus)

**What is it?**

Named Entity Recognition (NER) automatically finds and classifies proper nouns - people, organisations, locations, dates, and more.

This comes for free with spaCy - no extra code needed.

| Label | Meaning | Example |
|-------|---------|--------|
| PERSON | People's names | Tim Robbins, Morgan Freeman |
| GPE | Country, city, or state | American, California |
| ORG | Company or institution | Netflix, Warner Bros |
| DATE | Dates and time periods | 1994, last year |
| WORK_OF_ART | Titles of creative works | The Shawshank Redemption |

In [22]:
print("Named entities in the full review:")
print()
print(f"{"Entity Text":<35} {"Label":<15} {"Explanation"}")
print("-" * 75)

for ent in doc.ents:
    explanation = spacy.explain(ent.label_) or ""
    print(f"{ent.text:<35} {ent.label_:<15} {explanation}")

Named entities in the full review:

Entity Text                         Label           Explanation
---------------------------------------------------------------------------
Tim Robbins                         PERSON          People, including fictional
Morgan Freeman                      ORG             Companies, agencies, institutions, etc.
American                            NORP            Nationalities or religious or political groups


In [23]:
# displacy can highlight entities in the text with color
displacy.render(doc, style="ent", jupyter=True)

---
## Step 9 - Putting It All Together

Now we combine every step into a single reusable function.

This is what a production NLP preprocessing pipeline looks like.

In [24]:
def full_nlp_pipeline(raw_text):
    """Runs the complete NLP preprocessing pipeline on raw text."""

    # Step 1: basic cleaning
    text = re.sub(r"<.*?>", " ", raw_text)      # remove HTML tags
    text = re.sub(r"http\S+", "", text)          # remove URLs
    text = re.sub(r" +", " ", text).strip()      # collapse spaces

    # Step 2: spaCy processes everything in one call
    doc = nlp(text)

    # Step 3: sentences
    sentences = [sent.text.strip() for sent in doc.sents]

    # Step 4: noun phrases
    noun_phrases = [chunk.text for chunk in doc.noun_chunks]

    # Step 5: named entities
    entities = [(ent.text, ent.label_) for ent in doc.ents]

    # Step 6: lemmatized tokens ready for a model
    lemmas = [
        token.lemma_.lower()
        for token in doc
        if not token.is_punct
        and not token.is_space
        and not token.is_stop
    ]

    # Step 7: adjectives
    adjectives = [token.text for token in doc if token.pos_ == "ADJ"]

    return {
        "sentences":    sentences,
        "noun_phrases": noun_phrases,
        "entities":     entities,
        "lemmas":       lemmas,
        "adjectives":   adjectives,
    }


result = full_nlp_pipeline(raw_review)

print("Full Pipeline Output")
print("=" * 60)

print()
print("SENTENCES:")
for i, s in enumerate(result["sentences"], 1):
    print(f"  {i}. {s}")

print()
print("NOUN PHRASES:")
print(" ", result["noun_phrases"])

print()
print("NAMED ENTITIES:")
for text, label in result["entities"]:
    print(f"  {text:<30} [{label}]")

print()
print("ADJECTIVES:")
print(" ", result["adjectives"])

print()
print("LEMMAS (ready for ML model input):")
print(" ", result["lemmas"])

Full Pipeline Output

SENTENCES:
  1. The Shawshank Redemption is one of the greatest films ever made.
  2. Tim Robbins and Morgan Freeman deliver performances that are absolutely breathtaking.
  3. The story of hope and friendship in an American prison kept me on the edge of my seat.
  4. I loved every single scene.
  5. This movie deserves all the awards it has ever received!!!

NOUN PHRASES:
  ['The Shawshank Redemption', 'the greatest films', 'Tim Robbins', 'Morgan Freeman', 'performances', 'that', 'The story', 'hope', 'friendship', 'an American prison', 'me', 'the edge', 'my seat', 'I', 'every single scene', 'This movie', 'all the awards', 'it']

NAMED ENTITIES:
  Tim Robbins                    [PERSON]
  Morgan Freeman                 [ORG]
  American                       [NORP]

ADJECTIVES:
  ['greatest', 'breathtaking', 'American', 'single']

LEMMAS (ready for ML model input):
  ['shawshank', 'redemption', 'great', 'film', 'tim', 'robbins', 'morgan', 'freeman', 'deliver', 'per

---
## Step 10 - Compare a Positive and a Negative Review

Run the same pipeline on a negative review and compare the outputs.

Notice how the adjectives and lemmas differ completely. This difference is exactly what a sentiment classifier learns to detect.

In [25]:
negative_review = (
    "This film was a complete waste of time. "
    "The acting was terrible and the plot made absolutely no sense. "
    "I fell asleep halfway through and I have no intention of watching it again. "
    "Avoid this movie at all costs."
)

negative_result = full_nlp_pipeline(negative_review)

print("Negative Review Pipeline Output")
print("=" * 60)

print()
print("SENTENCES:")
for i, s in enumerate(negative_result["sentences"], 1):
    print(f"  {i}. {s}")

print()
print("ADJECTIVES:")
print(" ", negative_result["adjectives"])

print()
print("LEMMAS:")
print(" ", negative_result["lemmas"])

Negative Review Pipeline Output

SENTENCES:
  1. This film was a complete waste of time.
  2. The acting was terrible and the plot made absolutely no sense.
  3. I fell asleep halfway through and I have no intention of watching it again.
  4. Avoid this movie at all costs.

ADJECTIVES:
  ['complete', 'terrible', 'asleep']

LEMMAS:
  ['film', 'complete', 'waste', 'time', 'acting', 'terrible', 'plot', 'absolutely', 'sense', 'fall', 'asleep', 'halfway', 'intention', 'watch', 'avoid', 'movie', 'cost']


In [26]:
positive_adj = sorted(set(result["adjectives"]))
negative_adj = sorted(set(negative_result["adjectives"]))

print("Adjective comparison - Positive review vs Negative review:")
print()
print(f"{"Positive review":<35} {"Negative review"}")
print("-" * 65)

max_len = max(len(positive_adj), len(negative_adj))
for i in range(max_len):
    pos_word = positive_adj[i] if i < len(positive_adj) else ""
    neg_word = negative_adj[i] if i < len(negative_adj) else ""
    print(f"  {pos_word:<33} {neg_word}")

print()
print("A classifier sees breathtaking and greatest and predicts POSITIVE.")
print("A classifier sees terrible and complete and predicts NEGATIVE.")
print("This difference in words is the foundation of how sentiment analysis works.")

Adjective comparison - Positive review vs Negative review:

Positive review                     Negative review
-----------------------------------------------------------------
  American                          asleep
  breathtaking                      complete
  greatest                          terrible
  single                            

A classifier sees breathtaking and greatest and predicts POSITIVE.
A classifier sees terrible and complete and predicts NEGATIVE.
This difference in words is the foundation of how sentiment analysis works.


---
## Summary - What We Covered in Demo 1

We took a raw IMDb movie review and ran it through a complete NLP preprocessing pipeline.

| Step | What we did | spaCy tool |
|------|------------|----------|
| 0 | Loaded the English language model | `spacy.load("en_core_web_sm")` |
| 1 | Split the paragraph into sentences | `doc.sents` |
| 2 | Split each sentence into tokens | `token.text` |
| 3 | Cleaned text with regex | `re.sub(...)` |
| 4 | Tagged each word with its POS | `token.pos_` |
| 5 | Parsed grammatical dependencies | `token.dep_`, `token.head` |
| 6 | Extracted meaningful noun phrases | `doc.noun_chunks` |
| 7 | Reduced words to their base form | `token.lemma_` |
| 8 | Found named entities | `doc.ents` |
| 9 | Wrapped everything into one function | `full_nlp_pipeline()` |
| 10 | Compared positive vs negative output | saw adjective differences |

---

**Key takeaway:**

The final list of lemmas - clean, normalised, meaningful tokens - is what gets passed to a machine learning model. Everything in this notebook was preparation for that step.

---